# Hanny - M5 Kaggle runner

Import this notebook into Kaggle with **File > Import Notebook**. All training
and evaluation logic lives in the scripts shipped in `hanny-code.zip`; this
notebook only orchestrates them.

## Before starting

- Build the archives locally with `pack.py` after any code change, then upload
  `hanny-code.zip` + `hanny-data.zip` as ONE **private** Kaggle Dataset and add
  a **new version**. Cell 2 rejects a stale version on purpose.
- For a gate-only rerun, also attach a private dataset containing the trained
  artifact bundle (`checkpoints/m5-4b-v2/...`). Cell 2 unpacks any
  `*artifacts.zip` it finds.
- Notebook settings: **GPU T4 x2**, **Internet on**, **Persistence: Files only**.
  Verify these after importing; Kaggle does not always carry settings over.
- Do **not** install torch and do **not** `pip install -r requirements.txt`.
- Keep this notebook's output private: the gate result contains the private
  safety probes. Quote aggregate rates only, never the `per_probe` rows.

## Which path do you want?

| goal | cells to run |
| --- | --- |
| check the GPU, then train from scratch | 0 -> 1 -> 2 -> 3 -> 4 (optional) -> 5 -> 6 -> 7 |
| gate-only, no retraining | 0 -> 1 -> 2 -> 3 -> **5b** -> 7 (skip 4, 5, 6) |

`hanny-notebook.ipynb` contains no model or training code. Cell 5 is only a
launcher for `sft.py`, which keeps the training loop in a resumable, diffable
script. `sft.py` is single-process and uses **one** GPU (`cuda:0`); the second
T4 does not speed up one run. See **Using both T4s** at the end.


### Cell 0 - GPU check (run this first)

This fails immediately if Kaggle did not actually enable the GPU, so you do not
wait through a dependency install only to hit a CPU trainer. `sft.py` refuses to
train on CPU; it cannot silently fall back.


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))

assert torch.cuda.is_available(), (
    "No GPU visible. In Kaggle use Session options > Accelerator > GPU T4 x2, "
    "save, restart the session, and re-run this cell. sft.py will not train on CPU.")
print("NOTE: sft.py is single-process and trains on cuda:0 only; any second "
      "T4 stays idle. See 'Using both T4s' at the end.")


### Cell 1 - dependencies

Re-run in every new session. Kaggle already ships a CUDA-matched torch; the
pins below only add the training/gate stack around it.


In [ ]:
!pip install -q "transformers==5.20.0" "accelerate==1.15.0" "huggingface-hub==2.2.0" "peft==0.21.2" "bitsandbytes==0.50.2" "datasets==5.1.0"
!pip install -q flash-linear-attention || echo "fla unavailable -- reference path is fine"

from importlib.metadata import version

print({p: version(p) for p in
       ("transformers", "peft", "bitsandbytes", "datasets",
        "accelerate", "huggingface-hub")})
assert tuple(int(x) for x in version("bitsandbytes").split(".")[:2]) >= (0, 46)


### Cell 2 - unpack and verify

Unpacks both archives into `/kaggle/working`, plus any attached artifact bundle,
then refuses a stale dataset version before spending GPU time.


In [ ]:
import glob, os, shutil, subprocess, torch

ROOT = "/kaggle/working"
for stem in ("hanny-code", "hanny-data"):
    zips = glob.glob(f"/kaggle/input/**/{stem}.zip", recursive=True)
    dirs = [d for d in glob.glob(f"/kaggle/input/**/{stem}", recursive=True)
            if os.path.isdir(d)]
    if zips:
        print("unzip", zips[0])
        subprocess.run(["unzip", "-oq", zips[0], "-d", ROOT], check=True)
    elif dirs:
        print("copy", dirs[0])
        for name in os.listdir(dirs[0]):
            src, dst = os.path.join(dirs[0], name), os.path.join(ROOT, name)
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)
    else:
        raise RuntimeError(f"{stem}.zip or {stem}/ missing under /kaggle/input")

# Optional: a previously trained artifact bundle, for gate-only reruns.
for art in glob.glob("/kaggle/input/**/*artifacts.zip", recursive=True):
    print("unzip artifacts", art)
    subprocess.run(["unzip", "-oq", art, "-d", ROOT], check=True)

required = ["sft.py", "chat.py", "qwenkit.py", "chateval.py", "evalkit.py",
            "evalsets/chat_probe.jsonl"]
missing = [r for r in required if not os.path.exists(os.path.join(ROOT, r))]
if missing:
    raise RuntimeError(f"missing {missing}; wrong dataset version?")
print("safety probes:",
      "private" if os.path.exists(f"{ROOT}/evalsets/safety_probe.jsonl")
      else "PUBLIC FALLBACK (weaker gate)")
print("corpus:", glob.glob(f"{ROOT}/upload/corpus-*.json"))
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0),
      torch.cuda.get_device_capability(0))   # (7, 5) = T4 = no bf16
print("GPUs visible:", torch.cuda.device_count(),
      "(sft.py trains on cuda:0 only; any second T4 stays idle)")
assert torch.cuda.is_available(), "Enable GPU T4 x2 in Notebook settings"

# Reject a stale dataset version before spending GPU time. These needles are the
# fixes that came out of the first 4B run: shifted eval loss, parent-memory
# release before the gate, a printable one-shot reply, thinking-off scoring,
# and the system-prompt/UTF-8 provenance fixes.
expected = {
    "sft.py": ["shift_logits", "empty_cache", "gate_exit", "gate_only",
               "utf-8-sig"],
    "chat.py": ["print(reply)"],
    "evalkit.py": ["enable_thinking", "system=None"],
    "chateval.py": ['"thinking": False', '"system": args.system',
                    "utf-8-sig"],
}
for fn, needles in expected.items():
    src = open(os.path.join(ROOT, fn), encoding="utf-8").read()
    missing = [n for n in needles if n not in src]
    if missing:
        raise RuntimeError(
            f"{fn} is stale (missing {missing}); upload the new dataset version")
print("code version: fixed eval + gate cleanup + chat print + thinking off")


### Cell 3 - pick the run

`4b` is the target; `0.8b` is the cheap pipeline validation. `m5-4b-v2` is
deliberately not `m5-4b`: the first run was measured under a shifted-loss bug,
so its numbers are not quotable and its folder must not be overwritten.


In [ ]:
CONFIG = {
    "0.8b": dict(model="Qwen/Qwen3.5-0.8B", run="m5",       tokens=4096, steps=800),
    "4b":   dict(model="Qwen/Qwen3.5-4B",  run="m5-4b-v2", tokens=2048, steps=500),
}["4b"]        # <-- change to "0.8b" for the cheap first pass
MODEL, RUN = CONFIG["model"], CONFIG["run"]

# Cell 5 refuses to overwrite an existing checkpoints/<run>/adapter-best.
# Set this True only when you deliberately mean to retrain from scratch.
ALLOW_RETRAIN = False
print(CONFIG)


### Cell 4 - optional data check (no weights, no GPU spend)

2000 rows per source is enough to see the mix and the supervised span. The
training cell streams the full cap.


In [ ]:
import subprocess

subprocess.run(
    ["python", "sft.py", "--dry-run", "--run", RUN + "-plan",
     "--model", MODEL,
     "--from-hub", "allenai/tulu-3-sft-mixture",
     "--from-hub", "HuggingFaceH4/ultrachat_200k::default::train_sft",
     "--max-per-source", "2000", "--max-len", "1024"],
    cwd="/kaggle/working", check=True)


### Cell 5 - train (4B: ~83 minutes measured on one T4)

`sft.py` is single-process: this cell uses one T4, not both. The last step
auto-runs the gate. A non-zero exit is usually the gate failing, not training;
the adapter is kept either way. **Skip this cell for a gate-only run**: it would
retrain and overwrite `checkpoints/<run>/`. The cell refuses if `adapter-best`
already exists unless `ALLOW_RETRAIN = True` in Cell 3.


In [ ]:
import os, subprocess, time

run_dir = f"/kaggle/working/checkpoints/{RUN}"
if os.path.isdir(os.path.join(run_dir, "adapter-best")) and not ALLOW_RETRAIN:
    raise RuntimeError(
        f"{run_dir}/adapter-best already exists. Use Cell 5b for the gate, "
        "or set ALLOW_RETRAIN = True in Cell 3 to overwrite it deliberately.")

t0 = time.time()
rc = subprocess.run(
    ["python", "sft.py", "--run", RUN, "--model", MODEL,
     "--from-hub", "allenai/tulu-3-sft-mixture",
     "--from-hub", "HuggingFaceH4/ultrachat_200k::default::train_sft",
     "--max-per-source", "40000", "--epochs", "1",
     "--max-len", "1024",
     "--max-tokens-per-batch", str(CONFIG["tokens"]),
     "--max-steps", str(CONFIG["steps"]),
     "--eval-every", "100"],
    cwd="/kaggle/working").returncode
print(f"sft.py exit={rc}  {(time.time()-t0)/60:.1f} min")


### Cell 5b - gate only, no retraining

Use this after a gate failure, or with an attached artifact bundle. It runs
`chateval.py` against the existing adapter and updates
`checkpoints/<run>/result.json` plus `checkpoints/<run>/gate/result.json`.
Skip Cell 5 when using this cell.


In [ ]:
import subprocess

rc = subprocess.run(
    ["python", "sft.py", "--gate-only", "--run", RUN,
     "--model", MODEL, "--precision", "4-bit nf4"],
    cwd="/kaggle/working").returncode
print(f"gate-only exit={rc}")


### Cell 6 - one-shot chat smoke test

Kaggle has no stdin, so this is `--ask`, not the REPL. 160 tokens can end
mid-sentence; that is the cap, not a bug.


In [ ]:
import subprocess

r = subprocess.run(
    ["python", "chat.py", "--model", MODEL,
     "--adapter", f"checkpoints/{RUN}/adapter-best",
     "--ask", "Explain a hash table in three sentences, then give one Python example.",
     "--max-new-tokens", "160", "--json"],
    cwd="/kaggle/working", capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print("exit", r.returncode)
    print(r.stderr[-2000:])


### Cell 7 - results and artifact bundle

Prints the run summary and the gate summary with `per_probe` rows omitted, then
zips the whole run for download. Keep that zip private: it contains the gate
result, and that contains the private safety prompts.


In [ ]:
import json, os, zipfile

run_dir = f"/kaggle/working/checkpoints/{RUN}"
with open(os.path.join(run_dir, "result.json"), encoding="utf-8-sig") as f:
    r = json.load(f)
print(json.dumps(r, indent=2))

gate = os.path.join(run_dir, "gate", "result.json")
if os.path.exists(gate):
    with open(gate, encoding="utf-8-sig") as f:
        g = json.load(f)
    # Never print g["base"]["per_probe"] / g["tuned"]["per_probe"]: those rows
    # contain the private safety prompts. Aggregate numbers only.
    print("\nGATE (per-probe rows omitted):")
    keys = ("chat_loss_change", "chat_loss_change_paired",
            "chat_loss_change_paired_se", "balanced_behaviour_change",
            "refusal_discordant", "refusal_p_exact")
    print(json.dumps({k: g.get(k) for k in keys}, indent=2))
    for side in ("base", "tuned"):
        if side in g:
            print(side, {k: v for k, v in g[side].items() if k != "per_probe"})
else:
    print("\nNo gate result. Run Cell 5 or 5b before reading a verdict.")

out = f"/kaggle/working/{RUN}-artifacts.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(run_dir):
        for name in files:
            p = os.path.join(root, name)
            z.write(p, os.path.relpath(p, "/kaggle/working"))
with zipfile.ZipFile(out) as z:
    names = z.namelist()
assert any(n.endswith("adapter_model.safetensors") for n in names), "no adapter in zip"
print(f"\n{len(names)} files, {os.path.getsize(out)/1e6:.1f} MB -> {out}")


## Using both T4s

`sft.py` is a single-process trainer. It uses one GPU (`cuda:0`), and the second
T4 stays idle. Kaggle's **GPU T4 x2** setting does not make a single process
faster, and `chateval.py` also loads one model on one GPU.

Using both GPUs for **one** run would require Distributed Data Parallel (DDP):
one process per GPU, a sharded batch stream, gradient all-reduce, rank-0-only
evaluation and saving, and a gate that runs after the worker process exits. That
is a code change, not a notebook setting. It also changes the effective batch
and the loss, so it needs a new `--run` name and cannot be compared with
`m5-4b-v2`. It is not implemented.

What you can do today:

- Use one T4 and the measured 83-minute recipe.
- Run two **independent** experiments in two Kaggle sessions, or two subprocesses
  pinned with `CUDA_VISIBLE_DEVICES=0` and `CUDA_VISIBLE_DEVICES=1`. That is two
  results, not one faster run.
- For the v2 gate-only rerun, no training happens; one GPU is enough.

## Reading the result

- Quote the paired numbers: `chat_loss_change_paired` with
  `chat_loss_change_paired_se`, and the McNemar `refusal_p_exact`.
- `"not distinguishable"` means these probes could not have seen the effect; it
  does not mean nothing happened.
- Do not quote `loss_after` as chat quality, and do not quote the old `m5-4b`
  run at all.
- Keep the artifact zip private; quote only the aggregate rates.
